# Texture, ODF Evaluation, And Pole-Figure Inversion

This notebook demonstrates the current PyTex texture model:

- pole-figure synthesis from orientation sets
- inverse-pole-figure synthesis
- kernel-weighted ODF evaluation
- discrete pole-figure inversion through an explicit orientation dictionary

## Discrete Forward Model

For measurement directions $\mathbf{s}_m$ and dictionary orientations $g_j$, the
current inversion builds a forward matrix

$$
A_{mj} = \frac{1}{|\mathcal{H}|}\sum_{h\in\mathcal{H}} K\bigl(\angle(\mathbf{s}_m, g_j h)\bigr),
$$

then solves a regularized nonnegative least-squares problem over the dictionary
weights.


In [ ]:
import numpy as np

from pytex import (
    AcquisitionGeometry,
    BenchmarkManifest,
    CalibrationRecord,
    CrystalMap,
    CrystalPlane,
    DiffractionGeometry,
    EulerSet,
    ExperimentManifest,
    FrameDomain,
    FrameTransform,
    Handedness,
    InversePoleFigure,
    KernelSpec,
    KinematicSimulation,
    Lattice,
    MeasurementQuality,
    MillerIndex,
    ODF,
    Orientation,
    OrientationRelationship,
    OrientationSet,
    Phase,
    PhaseTransformationRecord,
    PoleFigure,
    ReferenceFrame,
    Rotation,
    ScatteringSetup,
    SymmetrySpec,
    TransformationVariant,
    ValidationManifest,
    VectorSet,
    WorkflowResultManifest,
    ZoneAxis,
    plot_odf,
    plot_orientations,
    plot_pole_figure,
    plot_symmetry_elements,
    plot_symmetry_orbit,
    plot_vector_set,
)


def make_context():
    crystal = ReferenceFrame(
        "crystal",
        FrameDomain.CRYSTAL,
        ("a", "b", "c"),
        Handedness.RIGHT,
    )
    specimen = ReferenceFrame(
        "specimen",
        FrameDomain.SPECIMEN,
        ("x", "y", "z"),
        Handedness.RIGHT,
    )
    map_frame = ReferenceFrame(
        "map",
        FrameDomain.MAP,
        ("i", "j", "k"),
        Handedness.RIGHT,
    )
    detector = ReferenceFrame(
        "detector",
        FrameDomain.DETECTOR,
        ("u", "v", "n"),
        Handedness.RIGHT,
    )
    lab = ReferenceFrame(
        "lab",
        FrameDomain.LABORATORY,
        ("X", "Y", "Z"),
        Handedness.RIGHT,
    )
    symmetry = SymmetrySpec.from_point_group("m-3m", reference_frame=crystal)
    lattice = Lattice(3.6, 3.6, 3.6, 90.0, 90.0, 90.0, crystal_frame=crystal)
    phase = Phase("fcc_demo", lattice=lattice, symmetry=symmetry, crystal_frame=crystal)
    return crystal, specimen, map_frame, detector, lab, phase


In [ ]:
crystal, specimen, *_ , phase = make_context()
symmetry = phase.symmetry

support = OrientationSet.from_euler_angles(
    np.array(
        [
            [0.0, 0.0, 0.0],
            [90.0, 0.0, 0.0],
            [35.0, 25.0, 10.0],
        ]
    ),
    crystal_frame=crystal,
    specimen_frame=specimen,
    symmetry=symmetry,
    phase=phase,
)

odf = ODF.from_orientations(
    support,
    weights=[4.0, 2.0, 1.0],
    kernel=KernelSpec(name="von_mises_fisher", halfwidth_deg=10.0),
)

query = Orientation(
    rotation=Rotation.identity(),
    crystal_frame=crystal,
    specimen_frame=specimen,
    symmetry=symmetry,
    phase=phase,
)
print("ODF(query) =", odf.evaluate(query))
print("Volume fraction within 15 deg =", odf.volume_fraction(query, max_angle_deg=15.0))


In [ ]:
poles = (
    CrystalPlane(miller=MillerIndex([1, 0, 0], phase=phase), phase=phase),
    CrystalPlane(miller=MillerIndex([1, 1, 1], phase=phase), phase=phase),
)

pole_figures = odf.reconstruct_pole_figures(
    poles,
    include_symmetry_family=False,
    antipodal=False,
)

inversion = ODF.invert_pole_figures(
    pole_figures,
    orientation_dictionary=support,
    kernel=odf.kernel,
    regularization=1e-8,
    include_symmetry_family=False,
)

print("Converged:", inversion.converged)
print("Estimated weights:", inversion.odf.normalized_weights)
print("Residual norm:", inversion.residual_norm)


In [ ]:
ipf = InversePoleFigure.from_orientations(
    support,
    [0.0, 0.0, 1.0],
    reduce_by_symmetry=True,
    antipodal=True,
)
print(ipf.crystal_directions)
